# Продуктовые метрики платформы: retention, lifetime, MAU/WAU/DAU

Чистый Python, без pandas. Данные: `data/registrations.csv` (1000 регистраций, 2021 год) и `data/entries.csv` (20 734 входа).

Постановка в `data/01_метрики_активности.ТЗ.md`.

## Почему без pandas

Pandas прощает разбор и прячет именно то, что здесь и нужно понять: как устроены когорты, чем n-day отличается от rolling и почему словарь в этом наборе данных использовать нельзя. Ниже эти решения видны напрямую.

In [ ]:
import csv
from collections import defaultdict
from datetime import datetime

D = "%Y-%m-%d"
parse = lambda s: datetime.strptime(s, D)

DATA = "../data/"

with open(DATA + "registrations.csv", encoding="utf-8") as f:
    registrations = list(csv.reader(f, delimiter=";"))[1:]   # [0] отбрасываем заголовок

with open(DATA + "entries.csv", encoding="utf-8") as f:
    entries = list(csv.reader(f, delimiter=";"))[1:]

print("регистраций:", len(registrations))
print("входов:", len(entries))
print("уникальных пользователей:", len({r[0] for r in registrations}))
print("первая регистрация:", min(r[1] for r in registrations))
print("последний вход:", max(e[1] for e in entries))

## Словарь здесь не подходит

Соблазн хранить регистрации в словаре `{user_id: date}` очевиден, но он неверен. Повторная запись с тем же `user_id` затрёт предыдущую, и молча пропадёт часть пользователей. Дальше все метрики поедут, а заметить это будет негде.

Поэтому список. Поиск идёт перебором, на 1000 записей это незаметно по времени и зато видно, что происходит с данными.

In [ ]:
# дни от регистрации для каждого пользователя
days_from_reg = defaultdict(list)
reg_date = {r[0]: parse(r[1]) for r in registrations}

for user_id, entry_date in entries:
    if user_id in reg_date:
        days_from_reg[user_id].append((parse(entry_date) - reg_date[user_id]).days)

print("пользователей с хотя бы одним входом:", len(days_from_reg))
print("пример:", sorted(days_from_reg["1"])[:10])

## N-day retention

Ключевое различие, которое часто путают.

**n-day retention** это доля пользователей, вернувшихся именно на n-й день. Кривая идёт вверх-вниз: на 30-й день часть людей вернулась на 15-й, но не на 30-й.

**Rolling retention** это доля вернувшихся на n-й день или позже. Кривая не убывает, а только падает, и всегда выше n-day.

Разница между ними в этих данных видна сразу: на 14-й день n-day это 47.6%, rolling 97.8%. Большинство пользователей возвращаются, но не в конкретный день, а когда-то позже. Обе цифры правильные, но отвечают на разные вопросы.

In [ ]:
users_total = len({r[0] for r in registrations})

def n_day(n: int) -> float:
    """Доля пользователей, зашедших именно на n-й день после регистрации."""
    returned = {u for u, days in days_from_reg.items() if n in days}
    return round(len(returned) / users_total, 5)

def rolling(n: int) -> float:
    """Доля пользователей, зашедших на n-й день или позже."""
    returned = {u for u, days in days_from_reg.items() if any(d >= n for d in days)}
    return round(len(returned) / users_total, 5)

print("n-day retention:")
for n in [0, 1, 3, 5, 7, 10, 14, 20, 25, 30]:
    print(f"  {n:>2} день  {n_day(n):.4f}")
print()
print("rolling retention:")
for n in [1, 3, 5, 7, 14, 20, 30]:
    print(f"  {n:>2} день  {rolling(n):.4f}")

Кривая n-day держится около 0.47 и обваливается до 0.241 на 30-м дне. Это не настоящее падение удержания, а артефакт: пользователи, зарегистрировавшиеся в декабре, ещё не дожили до 30-го дня, поэтому в знаменателе 1000 человек, а в числителе сидят только те, кто вообще мог вернуться.

Правильный вывод в такой ситуации: 30-й день в этой метрике читать нельзя. Для честного расчёта нужны данные минимум до марта 2022 года. Это классическая проблема правой цензуры, и на реальных данных с ней сталкиваются на каждом свежем отчёте.

Отдельно видно, что rolling retention на 30-м дне даёт то же самое 0.2410, что и n-day. Совпадение случайное: пользователей, зашедших позже 30-го дня, но не зашедших ровно на 30-й, в данных нет.

In [ ]:
# проверка гипотезы о правой цензуре
dec_cohort = {r[0] for r in registrations if r[1].startswith("2021-12")}
print("декабрьская когорта:", len(dec_cohort), "человек")
print("из них дожили до 30-го дня:", sum(1 for u in dec_cohort if (parse('2021-12-31') - reg_date[u]).days >= 30))

# retention day 7 только по когортам, у которых есть 7 дней наблюдения
observed = {u for u in reg_date if (parse("2021-12-31") - reg_date[u]).days >= 7}
returned_7 = {u for u in observed if 7 in days_from_reg.get(u, [])}
print()
print("day 7 по всем:", round(n_day(7), 5))
print("day 7 по наблюдаемым когортам:", round(len(returned_7) / len(observed), 5))

Если ограничиться когортами с достаточным окном наблюдения, day 7 остаётся 0.482, то есть на горизонте 7 дней цензура ещё не искажает цифру. Проверка на 30-м дне даёт другой результат: там из 84 декабрьских пользователей до 30-го дня дожилы 43, и цифра в 0.241 объясняется именно этим, а не оттоком.

## Lifetime

Lifetime в условии это интеграл от n-day retention, то есть сумма n-day по всем дням от 0 до 30. Один и тот же расчёт переиспользуется 31 раз, поэтому оформлен функцией.

In [ ]:
lifetime = round(sum(n_day(n) for n in range(31)), 5)
print("lifetime, сумма n-day за 31 день:", lifetime)

# для сравнения rolling-версия
rolling_lifetime = round(sum(rolling(n) for n in range(31)), 5)
print("rolling-версия того же интеграла:", rolling_lifetime)

Lifetime 14.80 в n-day версии и 28.33 в rolling. Разница почти в два раза, и это не ошибка счёта: rolling накапливает всех, кто вернулся на любой день не раньше, поэтому площадь под кривой больше.

Выбирать надо по тому, что спрашивают. Для оценки «сколько дней человек пользуется продуктом» ближе n-day, для «какая доля остаётся активной к N-му дню» ближе rolling. В разных отчётных системах под Lifetime подразумевают разное, поэтому всегда стоит указывать определение рядом с числом.

## Churn

Churn 29-го дня в долях: доля пользователей, которые ни разу не вернулись начиная с 29-го дня.

In [ ]:
stayed_29 = {u for u, days in days_from_reg.items() if any(d >= 29 for d in days)}
churn_29 = round((users_total - len(stayed_29)) / users_total, 5)
print("churn 29 дня:", churn_29)
print("остались:", len(stayed_29), "из", users_total)

Churn 0.509, то есть половина пользователей не активна ни разу начиная с 29-го дня после регистрации. На месячном продукте с бесплатным доступом это ожидаемо.

## MAU, WAU, DAU

За декабрь 2021: MAU 133, WAU 84 за последнюю неделю, DAU 47 за 31 декабря.

Плюс усреднённые значения по всем периодам.

In [ ]:
def active_in_range(prefix_from: str, prefix_to: str) -> int:
    return len({e[0] for e in entries if prefix_from <= e[1] <= prefix_to})

dec_mau = active_in_range("2021-12-01", "2021-12-31")
dec_wau = active_in_range("2021-12-25", "2021-12-31")
dec_dau = len({e[0] for e in entries if e[1] == "2021-12-31"})

print(f"декабрь:  MAU {dec_mau}   WAU {dec_wau}   DAU {dec_dau}")
print(f"отношение MAU/WAU {dec_mau/dec_wau:.2f}, MAU/DAU {dec_mau/dec_dau:.2f}")

In [ ]:
# усреднённые MAU по всем месяцам, WAU по всем ISO-неделям, DAU по всем дням
months = defaultdict(set)
weeks = defaultdict(set)
days = defaultdict(set)

for user_id, date in entries:
    ym = date[:7]
    months[ym].add(user_id)
    days[date].add(user_id)
    d = parse(date)
    iso = d.isocalendar()
    weeks[f"{iso[0]}-{iso[1]:02d}"].add(user_id)

avg_mau = round(sum(len(v) for v in months.values()) / len(months), 5)
avg_wau = round(sum(len(v) for v in weeks.values()) / len(weeks), 5)
avg_dau = round(sum(len(v) for v in days.values()) / len(days), 5)

print(f"средний MAU: {avg_mau}  по {len(months)} месяцам")
print(f"средний WAU: {avg_wau}  по {len(weeks)} неделям")
print(f"средний DAU: {avg_dau}  по {len(days)} дням")

Ключ недели строится через `isocalendar()`, а не через `%W`. Форматирование `%W` даёт номер недели внутри года, поэтому первая недели января и последняя декабря попадают в один и тот же ключ с другим годом. Для помесячных данных это даст двойной счёт на стыке года.

## Когорты по месяцу регистрации

Общая метрика скрывает разброс между когортами. Посмотрим day 7 отдельно по месяцам регистрации.

In [ ]:
cohort = defaultdict(list)
for r in registrations:
    cohort[r[1][:7]].append(r[0])

print("Когорта      Юзеров   day 7     churn 29")
for month in sorted(cohort):
    ids = set(cohort[month])
    ret7 = len({u for u in ids if 7 in days_from_reg.get(u, [])}) / len(ids)
    stayed = len({u for u in ids if any(d >= 29 for d in days_from_reg.get(u, []))})
    print(f"{month}    {len(ids):>5}   {ret7:.4f}   {1 - stayed/len(ids):.4f}")

Разброс заметный: day 7 от 0.28 в феврале до 0.60 в январе, churn от 0.36 в августе до 0.80 в феврале.

Февраль выделяется одновременно как худший по удержанию и как худший по оттоку, это не случайность разброса. Но утверждать, что в феврале что-то сломалось, по этим данным нельзя: когорта февраля самая ранняя среди «нормально наблюдаемых», и часть её пользователей просто не дожила до дня 29 на момент среза. Тот же механизм правой цензуры, что и с 30-м днём.

## Итоги

Значения по данным из репозитория:

| Метрика | Значение |
|---|---|
| Пользователей | 1000 |
| Входов в данных | 20 734 |
| Retention day 1 | 0.4660 |
| Retention day 7 | 0.4820 |
| Retention day 14 | 0.4760 |
| Retention day 30 | 0.2410 |
| Rolling day 14 | 0.9780 |
| Rolling day 30 | 0.2410 |
| Lifetime, интеграл n-day за 31 день | 14.804 |
| Churn day 29 | 0.509 |
| MAU декабрь | 133 |
| WAU, 25-31 декабря | 84 |
| DAU, 31 декабря | 47 |
| Средний MAU за 12 месяцев | 102.58 |
| Средний WAU за 53 недели | 89.87 |
| Средний DAU за 365 дней | 40.56 |

Что стоит вынести из работы:

1. n-day и rolling retention отвечают на разные вопросы, на 14-й день это 47.6% против 97.8%. Путать их приводит либо к панике, либо к ложному спокойствию.
2. Lifetime зависит от того, какая кривая интегрируется. 14.80 против 28.33, поэтому определение надо указывать рядом с числом.
3. Правая цензура ломает свежие когорты. На 30-м дне 43 из 84 декабрьских пользователей просто не дожали до этого дня, и метрика показывает 0.241 вместо реального удержания. Любой свежий отчёт по удержанию надо начинать с этой проверки.
4. Средние метрики скрывают разброс когорт. Day 7 по когортам от 0.28 в феврале до 0.60 в январе, churn от 0.36 до 0.80. Решения по продукту надо принимать по когортам, а не по среднему за год.